# Bank Transaction Fraud Detection - Exploratory Data Analysis

This notebook performs Phase 2 fraud analytics on the raw transaction dataset. The goal is to understand fraud patterns before model training: class imbalance, transaction amount behavior, feature relationships, outliers, and time-based signals.

Accuracy is not enough in fraud detection because fraudulent transactions are rare. A model that predicts every transaction as legitimate can look accurate while failing the business objective. The EDA below focuses on patterns that help improve recall, precision, and risk prioritization.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocess import clean_data, load_dataset, resolve_target_column

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
VIS_DIR = PROJECT_ROOT / 'visualizations'
VIS_DIR.mkdir(exist_ok=True)

## 1. Load Dataset

We load the raw transaction dataset and automatically resolve the fraud label column. In this sample dataset, the target is `fraud_label`, where `1` means fraud and `0` means legitimate.

In [ ]:
df = load_dataset(DATA_PATH)
target_col = resolve_target_column(df)
display(df.head())
print(f'Shape: {df.shape}')
print(f'Target column: {target_col}')

## 2. Statistical Summary

This gives the first operational view of the data: row count, missing values, duplicate rows, numeric ranges, and the raw fraud rate.

In [ ]:
summary = pd.DataFrame({
    'rows': [len(df)],
    'columns': [df.shape[1]],
    'duplicates': [df.duplicated().sum()],
    'missing_values': [df.isna().sum().sum()],
    'fraud_rate_pct': [df[target_col].mean() * 100],
})
display(summary)
display(df.describe(include='all').transpose())

## 3. Fraud vs Legitimate Class Distribution

Fraud datasets are imbalanced by nature. This chart shows why resampling and fraud-specific metrics are required.

In [ ]:
class_counts = df[target_col].value_counts().sort_index()
display(pd.DataFrame({'count': class_counts, 'percentage': class_counts / len(df) * 100}))

ax = sns.countplot(data=df, x=target_col, hue=target_col, palette='Set2')
ax.set_title('Fraud vs Legitimate Transactions')
ax.set_xlabel('Class: 0 = Legitimate, 1 = Fraud')
ax.set_ylabel('Transaction Count')
ax.legend_.remove()
plt.tight_layout()
plt.savefig(VIS_DIR / 'eda_class_distribution.png', dpi=160)
plt.show()

## 4. Transaction Amount Analysis

Transaction amount is often a major fraud signal. We compare legitimate and fraudulent amount distributions using summary statistics, histograms, and boxplots.

In [ ]:
if 'transaction_amount' in df.columns:
    display(df.groupby(target_col)['transaction_amount'].describe())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(data=df, x='transaction_amount', hue=target_col, bins=25, kde=True, ax=axes[0])
    axes[0].set_title('Transaction Amount Distribution')
    axes[0].set_xlabel('Transaction Amount')

    sns.boxplot(data=df, x=target_col, y='transaction_amount', hue=target_col, ax=axes[1])
    axes[1].set_title('Transaction Amount by Class')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Transaction Amount')
    axes[1].legend_.remove()

    plt.tight_layout()
    plt.savefig(VIS_DIR / 'eda_transaction_amount.png', dpi=160)
    plt.show()

## 5. Correlation Heatmap

Correlation does not prove causation, but it helps identify numeric features that move with the fraud label and deserve closer modeling attention.

In [ ]:
numeric_df = df.select_dtypes(include='number')
if numeric_df.shape[1] > 1:
    corr = numeric_df.corr(numeric_only=True)
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Numeric Feature Correlation Heatmap')
    plt.tight_layout()
    plt.savefig(VIS_DIR / 'eda_correlation_heatmap.png', dpi=160)
    plt.show()

    display(corr[[target_col]].sort_values(target_col, key=lambda s: s.abs(), ascending=False))

## 6. Feature Signal Exploration

After applying the same cleaning logic used by the production pipeline, we inspect which engineered or encoded features have the strongest linear relationship with fraud.

In [ ]:
cleaned_df = clean_data(df, target_column=target_col)
feature_signal = (
    cleaned_df.corr(numeric_only=True)[target_col]
    .drop(target_col)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .head(15)
)
display(feature_signal.to_frame('correlation_with_fraud'))

sns.barplot(x=feature_signal.values, y=feature_signal.index, palette='viridis')
plt.title('Top Feature Signals by Correlation With Fraud')
plt.xlabel('Correlation')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(VIS_DIR / 'eda_feature_signal.png', dpi=160)
plt.show()

## 7. Outlier Analysis

Outliers can represent either legitimate high-value behavior or suspicious activity. We flag amount outliers using the IQR method and compare their fraud rates.

In [ ]:
if 'transaction_amount' in df.columns:
    q1 = df['transaction_amount'].quantile(0.25)
    q3 = df['transaction_amount'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_frame = df.assign(is_amount_outlier=~df['transaction_amount'].between(lower, upper))
    display(outlier_frame.groupby('is_amount_outlier')[target_col].agg(['count', 'mean']))

    sns.scatterplot(
        data=outlier_frame.reset_index(),
        x='index',
        y='transaction_amount',
        hue=target_col,
        style='is_amount_outlier',
    )
    plt.axhline(upper, color='red', linestyle='--', label='Upper IQR threshold')
    plt.title('Transaction Amount Outlier Review')
    plt.xlabel('Transaction Index')
    plt.ylabel('Transaction Amount')
    plt.legend()
    plt.tight_layout()
    plt.savefig(VIS_DIR / 'eda_outliers.png', dpi=160)
    plt.show()

## 8. Time-Based Fraud Analysis

Fraud activity can cluster by hour, weekday, or weekend behavior. We extract these fields from the transaction timestamp and compare fraud rates.

In [ ]:
time_columns = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
if time_columns:
    time_col = time_columns[0]
    time_df = df.copy()
    time_df[time_col] = pd.to_datetime(time_df[time_col], errors='coerce')
    time_df['hour'] = time_df[time_col].dt.hour
    time_df['dayofweek'] = time_df[time_col].dt.dayofweek
    time_df['is_weekend'] = time_df['dayofweek'].isin([5, 6])

    hourly = time_df.groupby('hour')[target_col].agg(['count', 'mean']).reset_index()
    display(hourly)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=hourly, x='hour', y='mean', marker='o', ax=axes[0])
    axes[0].set_title('Fraud Rate by Hour')
    axes[0].set_ylabel('Fraud Rate')

    sns.countplot(data=time_df, x='dayofweek', hue=target_col, ax=axes[1])
    axes[1].set_title('Transactions by Day of Week')
    axes[1].set_xlabel('Day of Week: 0 = Monday')

    plt.tight_layout()
    plt.savefig(VIS_DIR / 'eda_time_patterns.png', dpi=160)
    plt.show()

## Key Fraud Analytics Takeaways

- Class imbalance means recall, precision, F1, ROC-AUC, and precision-recall curves are more useful than accuracy alone.
- Transaction amount, failed attempts, international behavior, device type, and time-derived features are useful risk signals to inspect.
- Outliers are not automatically fraud, but they are valuable for risk scoring and manual review thresholds.
- SMOTE must be applied only after the train/test split and only to the training data.